In [1]:
import pandas as pd
import os

import re

from Bio import SeqIO
import h5py

In [2]:
# ! pip install opentree
# ! pip install biopython
# ! pip install ete3
# ! pip install matplotlib
# ! pip install PyQt5
# ! pip install ete3

In [5]:
def read_chunk_from_hdf5(hdf5_path, dataset_name, start, length):
    with h5py.File(hdf5_path, 'r') as hdf5_file:
        dset = hdf5_file[dataset_name]
        chunk = dset[start:start + length].astype(str)
        return ''.join(chunk)

In [11]:
with h5py.File("mammals_gender_data/train.h5", 'r') as hdf5_file:
    print(hdf5_file['GCF_000001405.40'].keys())

<KeysViewHDF5 ['NC_000001.11 chromosome 1 ', 'NC_000002.12 chromosome 2 ', 'NC_000003.12 chromosome 3 ', 'NC_000004.12 chromosome 4 ', 'NC_000005.10 chromosome 5 ', 'NC_000006.12 chromosome 6 ', 'NC_000007.14 chromosome 7 ', 'NC_000008.11 chromosome 8 ', 'NC_000009.12 chromosome 9 ', 'NC_000010.11 chromosome 10 ', 'NC_000011.10 chromosome 11 ', 'NC_000012.12 chromosome 12 ', 'NC_000013.11 chromosome 13 ', 'NC_000014.9 chromosome 14 ', 'NC_000015.10 chromosome 15 ', 'NC_000016.10 chromosome 16 ', 'NC_000017.11 chromosome 17 ', 'NC_000018.10 chromosome 18 ', 'NC_000019.10 chromosome 19 ', 'NC_000020.11 chromosome 20 ', 'NC_000021.9 chromosome 21 ', 'NC_000022.11 chromosome 22 ', 'NC_000023.11 chromosome X ', 'NC_000024.10 chromosome Y ', 'NT_113793.3 chromosome 4 unlocalized genomic scaffold', 'NT_113796.3 chromosome 14 unlocalized genomic scaffold', 'NT_113888.1 chromosome 14 unlocalized genomic scaffold', 'NT_113891.3 chromosome 6 genomic scaffold', 'NT_113930.2 chromosome 17 unlocaliz

In [7]:
dir = '/mnt/20tb/vsfishman/nn_interpretator/mammals_fasta_dataset/dataset'

In [8]:
df = pd.read_csv(os.path.join(dir, 'mammals_dataset_metadata3.txt'), sep='\t')
df

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
0,GCF_004115215.2,Ornithorhynchus anatinus,21,5,20
1,GCF_015852505.1,Tachyglossus aculeatus,31,5,9
2,GCF_902635505.1,Sarcophilus harrisii,6,1,1
3,GCF_011100635.1,Trichosurus vulpecula,9,10,0
4,GCF_030445035.1,Dasypus novemcinctus,46,1,0
...,...,...,...,...,...
120,GCF_002776525.5,Piliocolobus tephrosceles,21,1,1
121,GCF_903992535.2,Arvicola amphibius,17,1,0
122,GCF_019320065.1,Cervus canadensis,33,1,1
123,GCF_028023285.1,Balaenoptera ricei,54,1,0


In [9]:
len(df[(df['number_of_x'] > 0) & (df['number_of_y'] > 0)])

60

In [10]:
df[(df['number_of_x'] != 0) & (df['number_of_y'] != 0)]

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
0,GCF_004115215.2,Ornithorhynchus anatinus,21,5,20
1,GCF_015852505.1,Tachyglossus aculeatus,31,5,9
2,GCF_902635505.1,Sarcophilus harrisii,6,1,1
7,GCF_020740605.2,Lemur catta,37,1,1
8,GCF_027406575.1,Nycticebus coucang,24,1,1
9,GCF_011100555.1,Callithrix jacchus,22,16,87
10,GCF_003339765.1,Macaca mulatta,307,5,1
11,GCF_008728515.1,Papio anubis,20,1,1
13,GCF_029289425.2,Pan paniscus,23,1,1
14,GCF_028885655.2,Pongo abelii,27,1,1


In [8]:
df = df[(df['number_of_x'] != 0) & (df['number_of_y'] != 0)] 
len(df)

60

In [9]:
len(df.organism_name.unique())

60

In [10]:
print("\n".join(df.organism_name))

Ornithorhynchus anatinus
Tachyglossus aculeatus
Sarcophilus harrisii
Lemur catta
Nycticebus coucang
Callithrix jacchus
Macaca mulatta
Papio anubis
Pan paniscus
Pongo abelii
Homo sapiens
Canis lupus familiaris
Ursus arctos
Lutra lutra
Meles meles
Mustela lutreola
Zalophus californianus
Delphinus delphis
Globicephala melas
Tursiops truncatus
Balaenoptera musculus
Loxodonta africana
Equus asinus
Sus scrofa
Camelus dromedarius
Bos javanicus
Bos taurus
Bos indicus
Ovis aries
Ochotona princeps
Lepus europaeus
Meriones unguiculatus
Mus musculus
Rattus norvegicus
Rattus rattus
Apodemus sylvaticus
Monodelphis domestica
Eubalaena glacialis
Choloepus didactylus
Neomonachus schauinslandi
Dama dama
Sciurus carolinensis
Mastomys coucha
Mustela erminea
Budorcas taxicolor
Mesoplodon densirostris
Jaculus jaculus
Arvicanthis niloticus
Lynx canadensis
Neofelis nebulosa
Phyllostomus discolor
Myotis daubentonii
Elephas maximus indicus
Suncus etruscus
Cynocephalus volans
Manis pentadactyla
Macaca thibetana 

In [11]:
species = list(df.organism_name)
len(set(species))

60

In [12]:
assembly_accession_list = list(df.assembly_accession)
assembly_accession_list

['GCF_004115215.2',
 'GCF_015852505.1',
 'GCF_902635505.1',
 'GCF_020740605.2',
 'GCF_027406575.1',
 'GCF_011100555.1',
 'GCF_003339765.1',
 'GCF_008728515.1',
 'GCF_029289425.2',
 'GCF_028885655.2',
 'GCF_000001405.40',
 'GCF_014441545.1',
 'GCF_023065955.2',
 'GCF_902655055.1',
 'GCF_922984935.1',
 'GCF_030435805.1',
 'GCF_009762305.2',
 'GCF_949987515.1',
 'GCF_963455315.1',
 'GCF_011762595.1',
 'GCF_009873245.2',
 'GCF_030014295.1',
 'GCF_016077325.2',
 'GCF_000003025.6',
 'GCF_036321535.1',
 'GCF_032452875.1',
 'GCF_002263795.3',
 'GCF_000247795.1',
 'GCF_016772045.2',
 'GCF_030435755.1',
 'GCF_033115175.1',
 'GCF_030254825.1',
 'GCF_000001635.27',
 'GCF_036323735.1',
 'GCF_011064425.1',
 'GCF_947179515.1',
 'GCF_027887165.1',
 'GCF_028564815.1',
 'GCF_015220235.1',
 'GCF_002201575.2',
 'GCF_033118175.1',
 'GCF_902686445.1',
 'GCF_008632895.1',
 'GCF_009829155.1',
 'GCF_023091745.1',
 'GCF_025265405.1',
 'GCF_020740685.1',
 'GCF_011762505.1',
 'GCF_007474595.2',
 'GCF_028018385.1'

In [13]:
chromosomes_filenames = ['x_chromosome.fasta', 'y_chromosome.fasta', 'autosomes.fasta']
for a_a in assembly_accession_list:
    species_path = os.path.join(dir, a_a)
    for chr_file in chromosomes_filenames:
        species_chr_path = os.path.join(species_path, chr_file)
        with open(species_chr_path, 'r') as f:
            for line in f:
                if line.startswith(">"):
                    print(line)

>NC_041749.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X1, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041750.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X2, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041751.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X3, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041752.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X4, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041753.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X5, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053175.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y1, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053176.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y2, mOrnAna1.pri.v4, whole genome shotgun sequence

>NW_024396656.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y2 unlocalized genomic scaffold, mOrnAna1.pri.v4 scaffold_128_arrow_ctg1, whole genome shotgun sequence

>NW_0243

In [21]:
headers = []
chromosomes_filenames = ['x_chromosome.fasta', 'y_chromosome.fasta', 'autosomes.fasta']
for a_a in assembly_accession_list:
    species_path = os.path.join(dir, a_a)
    for chr_file in chromosomes_filenames:
        species_chr_path = os.path.join(species_path, chr_file)
        with open(species_chr_path, 'r') as f:
            for line in f:
                if line.startswith(">"):
                    headers.append(line)

In [37]:
chromosomes = []
descriptions = []
for line in headers:
    chr, desc = re.findall("(chromosome [A-Z]?\d*)([^,]*)", line)[0]
    if desc =='':
        print(line)
    chromosomes.append(chr)
    descriptions.append(desc)

>NC_041749.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X1, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041750.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X2, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041751.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X3, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041752.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X4, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_041753.1 Ornithorhynchus anatinus isolate Pmale09 chromosome X5, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053175.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y1, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053176.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y2, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053177.1 Ornithorhynchus anatinus isolate Pmale09 chromosome Y3, mOrnAna1.pri.v4, whole genome shotgun sequence

>NC_053178.1 Ornithorhynchus anatinus isolate Pmale09 chromosome

In [35]:
set(chromosomes)

{'chromosome 1',
 'chromosome 10',
 'chromosome 11',
 'chromosome 12',
 'chromosome 13',
 'chromosome 14',
 'chromosome 15',
 'chromosome 16',
 'chromosome 17',
 'chromosome 18',
 'chromosome 19',
 'chromosome 2',
 'chromosome 20',
 'chromosome 21',
 'chromosome 22',
 'chromosome 23',
 'chromosome 24',
 'chromosome 25',
 'chromosome 26',
 'chromosome 27',
 'chromosome 28',
 'chromosome 29',
 'chromosome 3',
 'chromosome 30',
 'chromosome 31',
 'chromosome 32',
 'chromosome 33',
 'chromosome 34',
 'chromosome 35',
 'chromosome 36',
 'chromosome 37',
 'chromosome 38',
 'chromosome 4',
 'chromosome 5',
 'chromosome 6',
 'chromosome 7',
 'chromosome 8',
 'chromosome 9',
 'chromosome A1',
 'chromosome A2',
 'chromosome A3',
 'chromosome B1',
 'chromosome B2',
 'chromosome B3',
 'chromosome B4',
 'chromosome C1',
 'chromosome C2',
 'chromosome D1',
 'chromosome D2',
 'chromosome D3',
 'chromosome D4',
 'chromosome E1',
 'chromosome E2',
 'chromosome E3',
 'chromosome F1',
 'chromosome F2',
 

In [38]:
set(descriptions)

{'',
 ' genomic patch of type FIX',
 ' genomic patch of type NOVEL',
 ' genomic scaffold',
 ' unlocalized genomic scaffold'}

In [16]:
df

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
0,GCF_004115215.2,Ornithorhynchus anatinus,21,5,20
1,GCF_015852505.1,Tachyglossus aculeatus,31,5,9
2,GCF_902635505.1,Sarcophilus harrisii,6,1,1
7,GCF_020740605.2,Lemur catta,37,1,1
8,GCF_027406575.1,Nycticebus coucang,24,1,1
9,GCF_011100555.1,Callithrix jacchus,22,16,87
10,GCF_003339765.1,Macaca mulatta,307,5,1
11,GCF_008728515.1,Papio anubis,20,1,1
13,GCF_029289425.2,Pan paniscus,23,1,1
14,GCF_028885655.2,Pongo abelii,27,1,1


In [18]:
test_species = ["Sarcophilus harrisii", "Monodelphis domestica", "Manis pentadactyla", "Equus asinus", "Sus scrofa", "Camelus dromedarius", 
                "Phyllostomus discolor", "Myotis daubentonii", "Suncus etruscus", "Lepus europaeus", "Ochotona princeps", 
                "Sciurus carolinensis", "Nycticebus coucang", "Lemur catta", "Cynocephalus volans", 'Elephas maximus indicus', 'Loxodonta africana',
                "Choloepus didactylus", "Ornithorhynchus anatinus", "Tachyglossus aculeatus"]

valid_species = ["Lynx canadensis", "Neofelis nebulosa", "Eubalaena glacialis", "Balaenoptera musculus", "Cervus canadensis", "Dama dama",
                 "Meriones unguiculatus", "Chionomys nivalis", "Jaculus jaculus", "Callithrix jacchus"]

train_species = [species for species in df.organism_name if species not in test_species and species not in valid_species]

In [19]:
assert len(test_species) == 20
assert len(valid_species) == 10
assert len(train_species) == 30

In [22]:
train_df = df[df['organism_name'].apply(lambda x: (x not in valid_species) and (x not in test_species))]
train_df

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
10,GCF_003339765.1,Macaca mulatta,307,5,1
11,GCF_008728515.1,Papio anubis,20,1,1
13,GCF_029289425.2,Pan paniscus,23,1,1
14,GCF_028885655.2,Pongo abelii,27,1,1
15,GCF_000001405.40,Homo sapiens,557,15,6
18,GCF_014441545.1,Canis lupus familiaris,38,1,3
19,GCF_023065955.2,Ursus arctos,0,1,1
21,GCF_902655055.1,Lutra lutra,18,1,1
22,GCF_922984935.1,Meles meles,21,4,1
23,GCF_030435805.1,Mustela lutreola,21,1,2


In [23]:
len(train_df)

30

In [21]:
with open(dir+"/"+assembly_accession_list[0]+"/autosomes.fasta") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        print(record.description)

NC_041728.1 Ornithorhynchus anatinus isolate Pmale09 chromosome 1, mOrnAna1.pri.v4, whole genome shotgun sequence
NC_041729.1 Ornithorhynchus anatinus isolate Pmale09 chromosome 2, mOrnAna1.pri.v4, whole genome shotgun sequence
NC_041730.1 Ornithorhynchus anatinus isolate Pmale09 chromosome 3, mOrnAna1.pri.v4, whole genome shotgun sequence
NC_041731.1 Ornithorhynchus anatinus isolate Pmale09 chromosome 4, mOrnAna1.pri.v4, whole genome shotgun sequence


KeyboardInterrupt: 